# Concordance 
* Notebooks
   * [391_concordance_SAT_ID_ref_stockholmarchipelagotrail.ipynb](https://github.com/salgo60/Stockholm_Archipelago_Trail/blob/main/Notebook/391_concordance_SAT_ID_ref_stockholmarchipelagotrail.ipynb)
   * [505_concordance.ipynb](https://github.com/salgo60/Stockholm_Archipelago_Trail/blob/main/Notebook/505_concordance.ipynb)

* [#505](https://github.com/salgo60/Stockholm_Archipelago_Trail/issues/505)

In [1]:
import time
import datetime  
start_time = time.time()
start_str = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
print(f"Started: {start_str}")


Started: 2026-09-15 02:33


In [25]:
import requests
from urllib.parse import quote
import pandas as pd

CONCORDANCE_URL = (
    "https://map.stockholmarchipelagotrail.com/"
    "data/geojson/poi-concordance.json"
)

API_URL = (
    "https://map.stockholmarchipelagotrail.com/"
    "api/objects/"
)

session = requests.Session()


# ============================================================
# 1. Read concordance
# ============================================================

data = session.get(
    CONCORDANCE_URL,
    timeout=30
).json()

sat_id_of = data["satIdOf"]

sat_ids = list(sat_id_of.values())

print(f"SAT objects: {len(sat_ids)}")


# ============================================================
# 2. Query API
# ============================================================

api_objects = {}

for i, sat_id in enumerate(sat_ids, 1):

    url = API_URL + quote(sat_id, safe="")

    try:
        response = session.get(
            url,
            timeout=20
        )

        if response.status_code == 200:
            api_objects[sat_id] = response.json()

        else:
            api_objects[sat_id] = {
                "_api_error": response.status_code
            }

    except Exception as e:

        api_objects[sat_id] = {
            "_api_error": str(e)
        }

    if i % 100 == 0 or i == len(sat_ids):
        print(f"{i}/{len(sat_ids)}")

print("API check complete")  
# ============================================================
# 3. Status distribution
# ============================================================

statuses = pd.Series([
    obj.get("status")
    for obj in api_objects.values()
])

print(
    statuses
    .fillna("(no status)")
    .value_counts()
)

# ============================================================
# 4. Inspect the two known examples
# ============================================================

for sat_id in [
    "sat:poi:2dgqk",
    "sat:poi:237tx"
]:

    obj = api_objects.get(sat_id, {})

    print("\n" + "=" * 60)
    print(sat_id)

    print("status:",
          obj.get("status"))

    print("firstSeen:",
          obj.get("firstSeen"))

    print("alternateIdentifiers:",
          obj.get("alternateIdentifiers"))

    print("sameAs:",
          obj.get("sameAs"))

    print("wikidata:",
          obj.get("wikidata"))

    print("error:",
          obj.get("_api_error"))
def extract_links(obj):

    osm = []
    wikidata = []

    # --------------------------------------------------------
    # alternateIdentifiers
    # --------------------------------------------------------

    for identifier in obj.get(
        "alternateIdentifiers", []
    ) or []:

        identifier = str(identifier).strip()

        if identifier.startswith("osm:"):

            parts = identifier.split(":")

            if len(parts) == 3:

                osm_type = parts[1]
                osm_id = parts[2]

                if osm_type in [
                    "node",
                    "way",
                    "relation"
                ]:

                    osm.append(
                        f"[{osm_type}:{osm_id}]"
                        f"(https://www.openstreetmap.org/"
                        f"{osm_type}/{osm_id})"
                    )

        elif identifier.startswith("wikidata:"):

            qid = identifier.split(":", 1)[1]

            wikidata.append(
                f"[{qid}]"
                f"(https://www.wikidata.org/wiki/{qid})"
            )

    # --------------------------------------------------------
    # sameAs
    # --------------------------------------------------------

    for link in obj.get("sameAs", []) or []:

        link = str(link).strip()

        if "openstreetmap.org/" in link:
            osm.append(f"[OSM]({link})")

        elif "wikidata.org/wiki/" in link:

            qid = link.rstrip("/").split("/")[-1]

            wikidata.append(
                f"[{qid}]({link})"
            )

    # Remove duplicates
    osm = list(dict.fromkeys(osm))
    wikidata = list(dict.fromkeys(wikidata))

    return "<br>".join(osm), "<br>".join(wikidata)  
import requests
from urllib.parse import quote
import pandas as pd

CONCORDANCE_URL = (
    "https://map.stockholmarchipelagotrail.com/"
    "data/geojson/poi-concordance.json"
)

API_URL = (
    "https://map.stockholmarchipelagotrail.com/"
    "api/objects/"
)

session = requests.Session()


# ============================================================
# 1. Read concordance
# ============================================================

data = session.get(
    CONCORDANCE_URL,
    timeout=30
).json()

sat_id_of = data["satIdOf"]

sat_ids = list(sat_id_of.values())

print(f"SAT objects: {len(sat_ids)}")


# ============================================================
# 2. Query API
# ============================================================

api_objects = {}

for i, sat_id in enumerate(sat_ids, 1):

    url = API_URL + quote(sat_id, safe="")

    try:
        response = session.get(
            url,
            timeout=20
        )

        if response.status_code == 200:
            api_objects[sat_id] = response.json()

        else:
            api_objects[sat_id] = {
                "_api_error": response.status_code
            }

    except Exception as e:

        api_objects[sat_id] = {
            "_api_error": str(e)
        }

    if i % 100 == 0 or i == len(sat_ids):
        print(f"{i}/{len(sat_ids)}")

print("API check complete")  
# ============================================================
# 3. Status distribution
# ============================================================

statuses = pd.Series([
    obj.get("status")
    for obj in api_objects.values()
])

print(
    statuses
    .fillna("(no status)")
    .value_counts()
)

# ============================================================
# 4. Inspect the two known examples
# ============================================================

for sat_id in [
    "sat:poi:2dgqk",
    "sat:poi:237tx"
]:

    obj = api_objects.get(sat_id, {})

    print("\n" + "=" * 60)
    print(sat_id)

    print("status:",
          obj.get("status"))

    print("firstSeen:",
          obj.get("firstSeen"))

    print("alternateIdentifiers:",
          obj.get("alternateIdentifiers"))

    print("sameAs:",
          obj.get("sameAs"))

    print("wikidata:",
          obj.get("wikidata"))

    print("error:",
          obj.get("_api_error"))
def extract_links(obj):

    osm = []
    wikidata = []

    # --------------------------------------------------------
    # alternateIdentifiers
    # --------------------------------------------------------

    for identifier in obj.get(
        "alternateIdentifiers", []
    ) or []:

        identifier = str(identifier).strip()

        if identifier.startswith("osm:"):

            parts = identifier.split(":")

            if len(parts) == 3:

                osm_type = parts[1]
                osm_id = parts[2]

                if osm_type in [
                    "node",
                    "way",
                    "relation"
                ]:

                    osm.append(
                        f"[{osm_type}:{osm_id}]"
                        f"(https://www.openstreetmap.org/"
                        f"{osm_type}/{osm_id})"
                    )

        elif identifier.startswith("wikidata:"):

            qid = identifier.split(":", 1)[1]

            wikidata.append(
                f"[{qid}]"
                f"(https://www.wikidata.org/wiki/{qid})"
            )

    # --------------------------------------------------------
    # sameAs
    # --------------------------------------------------------

    for link in obj.get("sameAs", []) or []:

        link = str(link).strip()

        if "openstreetmap.org/" in link:
            osm.append(f"[OSM]({link})")

        elif "wikidata.org/wiki/" in link:

            qid = link.rstrip("/").split("/")[-1]

            wikidata.append(
                f"[{qid}]({link})"
            )

    # Remove duplicates
    osm = list(dict.fromkeys(osm))
    wikidata = list(dict.fromkeys(wikidata))

    return "<br>".join(osm), "<br>".join(wikidata)  
# ============================================================
# 3. Build Markdown table
# ============================================================

rows = []

for sat_id, obj in api_objects.items():

    sat_link = (
        f"[{sat_id}]"
        f"(https://map.stockholmarchipelagotrail.com/"
        f"?poi={quote(sat_id, safe='')})"
    )

    json_link = (
        f"[JSON]"
        f"(https://map.stockholmarchipelagotrail.com/"
        f"api/objects/{quote(sat_id, safe='')})"
    )

    osm, wikidata = extract_links(obj)

    rows.append({
        "SAT ID": sat_link,
        "JSON": json_link,
        "Status": obj.get("status", ""),
        "firstSeen": obj.get("firstSeen", ""),
        "OSM": osm,
        "Wikidata": wikidata
    })


result = pd.DataFrame(rows)

SAT objects: 1143
100/1143
200/1143
300/1143
400/1143
500/1143
600/1143
700/1143
800/1143
900/1143
1000/1143
1100/1143
1143/1143
API check complete
(no status)    783
missing         38
Name: count, dtype: int64

sat:poi:2dgqk
status: None
firstSeen: 2026-08-24
alternateIdentifiers: ['osm:way:316767169']
sameAs: ['https://www.openstreetmap.org/way/316767169']
wikidata: None
error: None

sat:poi:237tx
status: missing
firstSeen: 2026-06-10
alternateIdentifiers: ['osm:way:857838618', 'wikidata:Q133864018']
sameAs: None
wikidata: None
error: None
SAT objects: 1143
100/1143
200/1143
300/1143
400/1143
500/1143
600/1143
700/1143
800/1143
900/1143
1000/1143
1100/1143
1143/1143
API check complete
(no status)    783
missing         38
Name: count, dtype: int64

sat:poi:2dgqk
status: None
firstSeen: 2026-08-24
alternateIdentifiers: ['osm:way:316767169']
sameAs: ['https://www.openstreetmap.org/way/316767169']
wikidata: None
error: None

sat:poi:237tx
status: missing
firstSeen: 2026-06-10
alternate

In [26]:
# ============================================================
# 7. Filter: NEW
# ============================================================

NEW_SINCE = "2026-09-14"

new = result[
    result["firstSeen"].fillna("") >= NEW_SINCE
].copy()

show_table(
    new,
    f"New objects since {NEW_SINCE}"
)

print(f"New objects: {len(new)}")

NameError: name 'show_table' is not defined


sat:poi:2dgqk
status: None
firstSeen: 2026-08-24
alternateIdentifiers: ['osm:way:316767169']
sameAs: ['https://www.openstreetmap.org/way/316767169']
wikidata: None
error: None

sat:poi:237tx
status: missing
firstSeen: 2026-06-10
alternateIdentifiers: ['osm:way:857838618', 'wikidata:Q133864018']
sameAs: None
wikidata: None
error: None


In [9]:
print(
    result["Status"]
    .replace("", "(no status)")
    .value_counts()
)

Status
(no status)    783
missing         38
Name: count, dtype: int64


KeyError: 'object'

In [17]:
print(type(result["JSON"].iloc[0]))

<class 'str'>
